In [0]:
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import date


schema = StructType([
    StructField("Team_1", StringType(), True),
    StructField("Team_2", StringType(), True),
    StructField("Winner", StringType(), True)
])

# Define the data
data = [
    ('India', 'SL', 'India'),
    ('SL', 'Aus', 'Aus'),
    ('SA', 'Eng', 'Eng'),
    ('Eng', 'NZ', 'NZ'),
    ('Aus', 'India', 'India')
]
spark = SparkSession.builder.appName("Customer Orders").getOrCreate()

# Create the DataFrame with the defined schema
icc_world_cup_df = spark.createDataFrame(data, schema=schema)

# Show the DataFrame
icc_world_cup_df.display()


Team_1,Team_2,Winner
India,SL,India
SL,Aus,Aus
SA,Eng,Eng
Eng,NZ,NZ
Aus,India,India


In [0]:
team1_df=icc_world_cup_df.select(col("Team_1").alias("team"),when(col("team_1")==col("winner"),1).otherwise(0).alias("win_flag"))
team2_df = icc_world_cup_df.select(
    col("Team_2").alias("team"),
    when(col("Team_2") == col("Winner"), 1).otherwise(0).alias("win_flag")
)
team_union=team1_df.union(team2_df)
team_union.groupBy("team").agg(
    count("*").alias("total_matches"),
    sum("win_flag").alias("total_wins"),
    count("*")-sum("win_flag").alias("total_losses")
).orderBy(col("total_wins").desc()).display()


team,total_matches,total_wins,(count(1) - sum(win_flag) AS total_losses)
India,2,2,0
Eng,2,1,1
Aus,2,1,1
NZ,1,1,0
SL,2,0,2
SA,1,0,1


In [0]:

# Define the schema using StructType
schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("order_date", DateType(), True),
    StructField("order_amount", IntegerType(), True)
])

# Create the data
data = [
    (1, 100, date(2022, 1, 1), 2000),
    (2, 200, date(2022, 1, 1), 2500),
    (3, 300, date(2022, 1, 1), 2100),
    (4, 100, date(2022, 1, 2), 2000),
    (5, 400, date(2022, 1, 2), 2200),
    (6, 500, date(2022, 1, 2), 2700),
    (7, 100, date(2022, 1, 3), 3000),
    (8, 400, date(2022, 1, 3), 1000),
    (9, 600, date(2022, 1, 3), 3000)
]
customer_orders_df = spark.createDataFrame(data, schema=schema)
customer_orders_df.show()


+--------+-----------+----------+------------+
|order_id|customer_id|order_date|order_amount|
+--------+-----------+----------+------------+
|       1|        100|2022-01-01|        2000|
|       2|        200|2022-01-01|        2500|
|       3|        300|2022-01-01|        2100|
|       4|        100|2022-01-02|        2000|
|       5|        400|2022-01-02|        2200|
|       6|        500|2022-01-02|        2700|
|       7|        100|2022-01-03|        3000|
|       8|        400|2022-01-03|        1000|
|       9|        600|2022-01-03|        3000|
+--------+-----------+----------+------------+



In [0]:
window=Window.partitionBy(col("customer_id"))
min_date=customer_orders_df.withColumn("min_date",min(col("order_date")).over(window))
flag_col=min_date.withColumn("new_flag",when(col("order_date")==col("min_date")
                                         ,1).otherwise(0)).withColumn("old_flag",when(col("order_date")!=col("min_date"),1).otherwise(0))
result=flag_col.groupBy("order_date").agg(sum("new_flag").alias("new_orders"),sum("old_flag").alias("old_orders")).orderBy(col("order_date"))
result.show()


+----------+----------+----------+
|order_date|new_orders|old_orders|
+----------+----------+----------+
|2022-01-01|         3|         0|
|2022-01-02|         2|         1|
|2022-01-03|         1|         2|
+----------+----------+----------+



In [0]:
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import date


# Define the schema
schema = StructType([
    StructField("name", StringType(), True),
    StructField("address", StringType(), True),
    StructField("email", StringType(), True),
    StructField("floor", IntegerType(), True),
    StructField("resources", StringType(), True)
])

# Define the data
data = [
    ('A', 'Bangalore', 'A@gmail.com', 1, 'CPU'),
    ('A', 'Bangalore', 'A1@gmail.com', 1, 'CPU'),
    ('A', 'Bangalore', 'A2@gmail.com', 2, 'DESKTOP'),
    ('B', 'Bangalore', 'B@gmail.com', 2, 'DESKTOP'),
    ('B', 'Bangalore', 'B1@gmail.com', 2, 'DESKTOP'),
    ('B', 'Bangalore', 'B2@gmail.com', 1, 'MONITOR')
]

# Create the DataFrame
entries_df = spark.createDataFrame(data, schema=schema)

# Show the DataFrame
entries_df.show()

+----+---------+------------+-----+---------+
|name|  address|       email|floor|resources|
+----+---------+------------+-----+---------+
|   A|Bangalore| A@gmail.com|    1|      CPU|
|   A|Bangalore|A1@gmail.com|    1|      CPU|
|   A|Bangalore|A2@gmail.com|    2|  DESKTOP|
|   B|Bangalore| B@gmail.com|    2|  DESKTOP|
|   B|Bangalore|B1@gmail.com|    2|  DESKTOP|
|   B|Bangalore|B2@gmail.com|    1|  MONITOR|
+----+---------+------------+-----+---------+



In [0]:
total_info= entries_df.groupBy("name").agg(count("*").alias("total_visit"),concat_ws(",",collect_set(col("resources"))).alias("resource"))
cout_of_floor_visit=entries_df.groupBy("name","floor").agg(count("*").alias("floor_count"))
w=Window.partitionBy("name").orderBy(col("floor_count").desc())
most_visit_floor=cout_of_floor_visit.withColumn("rank",dense_rank().over(w)).filter(col("rank")==1).select(col("name"),col("floor").alias("most_visited_floor"))
final=most_visit_floor.join(total_info,on="name").select(col("name"),col("most_visited_floor"),col("total_visit"),col("resource"))
final.show()

+----+------------------+-----------+---------------+
|name|most_visited_floor|total_visit|       resource|
+----+------------------+-----------+---------------+
|   A|                 1|          3|    CPU,DESKTOP|
|   B|                 2|          3|DESKTOP,MONITOR|
+----+------------------+-----------+---------------+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

data = [("2024-09-24 13:45:26",)]
df = spark.createDataFrame(data, ["timestamp_str"])
df = df.withColumn("timestamp", col("timestamp_str").cast("timestamp"))

df.select(
    "timestamp",
    date_trunc("hour", "timestamp").alias("trunc_hour"),
    date_trunc("day", "timestamp").alias("trunc_day"),
    date_trunc("month", "timestamp").alias("trunc_month"),
    date_trunc("quarter", "timestamp").alias("trunc_to_quarter"),
    date_trunc("year", "timestamp").alias("trunc_to_year")
).show(truncate=False)


+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+
|timestamp          |trunc_hour         |trunc_day          |trunc_month        |trunc_to_quarter   |trunc_to_year      |
+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+
|2024-09-24 13:45:26|2024-09-24 13:00:00|2024-09-24 00:00:00|2024-09-01 00:00:00|2024-07-01 00:00:00|2024-01-01 00:00:00|
+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+



In [0]:
df.withColumn("current_date", current_date()).withColumn("current_timestamp", current_timestamp()).show()
df = df.withColumn("timestamp", to_timestamp("timestamp"))
df.select(
    year("timestamp").alias("year"),
    month("timestamp").alias("month"),
    dayofmonth("timestamp").alias("day"),
    dayofweek("timestamp").alias("weekday"),  # 1 = Sunday
    dayofyear("timestamp").alias("day_of_year"),
    weekofyear("timestamp").alias("week"),
    quarter("timestamp").alias("quarter"),
    hour("timestamp").alias("hour"),
    minute("timestamp").alias("minute"),
    second("timestamp").alias("second")
).show()


+-------------------+-------------------+------------+--------------------+
|      timestamp_str|          timestamp|current_date|   current_timestamp|
+-------------------+-------------------+------------+--------------------+
|2024-07-24 13:45:26|2024-07-24 13:45:26|  2025-07-24|2025-07-24 12:00:...|
+-------------------+-------------------+------------+--------------------+

+----+-----+---+-------+-----------+----+-------+----+------+------+
|year|month|day|weekday|day_of_year|week|quarter|hour|minute|second|
+----+-----+---+-------+-----------+----+-------+----+------+------+
|2024|    7| 24|      4|        206|  30|      3|  13|    45|    26|
+----+-----+---+-------+-----------+----+-------+----+------+------+



In [0]:
df.select(
    to_date("timestamp").alias("as_date"),
    date_format("timestamp", "yyyy-MM-dd HH:mm").alias("formatted"),
    unix_timestamp("timestamp").alias("unix_time"),
    from_unixtime(unix_timestamp("timestamp")).alias("from_unix")
).show()


+----------+----------------+----------+-------------------+
|   as_date|       formatted| unix_time|          from_unix|
+----------+----------------+----------+-------------------+
|2024-07-24|2024-07-24 13:45|1721828726|2024-07-24 13:45:26|
+----------+----------------+----------+-------------------+



In [0]:
df.select(
    date_add("timestamp", 7).alias("add_7_days"),
    date_sub("timestamp", 7).alias("sub_7_days"),
    add_months("timestamp", 2).alias("add_2_months"),
    months_between(current_date(), "timestamp").alias("months_diff"),
    datediff(current_date(), to_date("timestamp")).alias("days_diff"),
    next_day("timestamp", "Friday").alias("next_friday"),
    last_day("timestamp").alias("end_of_month")
).show()


+----------+----------+------------+-----------+---------+-----------+------------+
|add_7_days|sub_7_days|add_2_months|months_diff|days_diff|next_friday|end_of_month|
+----------+----------+------------+-----------+---------+-----------+------------+
|2024-10-01|2024-09-17|  2024-11-24|       10.0|      303| 2024-09-27|  2024-09-30|
+----------+----------+------------+-----------+---------+-----------+------------+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
from decimal import Decimal
from pyspark.sql.window import Window


# Define schema with DecimalType for sales_amount

schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product_name", StringType(), True),
    StructField("sales_amount", FloatType(), True)
])

# Define data
data = [
    (101, 'Product C', 520.00),
    (102, 'Product D', 600.00),
    (103, 'Product A', 900.00),
    (104, 'Product B', 300.00),
    (105, 'Product q', 550.00),
    (106, 'Product F', 200.00),
    (107, 'Product G', 150.00),
    (108, 'Product H', 100.00),
    (109, 'Product I', 80.50),
    (110, 'Product J', 70.00),
]

# Create DataFrame
product_sales_df = spark.createDataFrame(data, schema=schema)
# Show DataFrame
product_sales_df.show()


+----------+------------+------------+
|product_id|product_name|sales_amount|
+----------+------------+------------+
|       101|   Product C|       520.0|
|       102|   Product D|       600.0|
|       103|   Product A|       900.0|
|       104|   Product B|       300.0|
|       105|   Product q|       550.0|
|       106|   Product F|       200.0|
|       107|   Product G|       150.0|
|       108|   Product H|       100.0|
|       109|   Product I|        80.5|
|       110|   Product J|        70.0|
+----------+------------+------------+



In [0]:
product_wise_sales=product_sales_df.groupBy("product_id").agg(sum("sales_amount").alias("total_sales"))
total_sale_amount = product_wise_sales.select(sum("total_sales")).collect()[0][0]
w=Window.orderBy(col("total_sales").desc()).rowsBetween (Window.unboundedPreceding,Window.currentRow)
running_sales=product_wise_sales.withColumn("80%oftotal", lit(0.8*total_sale_amount)).withColumn("running_sal",sum(col("total_sales")).over(w)).filter(col("running_sal")<=col("80%oftotal")).select(col("product_id"),col("total_sales")).show()

+----------+-----------+
|product_id|total_sales|
+----------+-----------+
|       103|      900.0|
|       102|      600.0|
|       105|      550.0|
|       101|      520.0|
+----------+-----------+



In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType

# Define schema
friend_schema = StructType([
    StructField("pid", IntegerType(), True),
    StructField("fid", IntegerType(), True)
])

# Insert data
friend_data = [
    (1, 2),
    (1, 3),
    (2, 1),
    (2, 3),
    (3, 5),
    (4, 2),
    (4, 3),
    (4, 5)
]

# Create DataFrame
friend_df = spark.createDataFrame(friend_data, schema=friend_schema)
friend_df.show()


+---+---+
|pid|fid|
+---+---+
|  1|  2|
|  1|  3|
|  2|  1|
|  2|  3|
|  3|  5|
|  4|  2|
|  4|  3|
|  4|  5|
+---+---+



In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Define schema
person_schema = StructType([
    StructField("PersonID", IntegerType(), True),
    StructField("Name", StringType(), True),
    StructField("Score", IntegerType(), True)
])

# Insert data
person_data = [
    (1, 'Alice', 88),
    (2, 'Bob', 11),
    (3, 'Devis', 27),
    (4, 'Tara', 45),
    (5, 'John', 63)
]

# Create DataFrame
person_df = spark.createDataFrame(person_data, schema=person_schema)
person_df.show()



+--------+-----+-----+
|PersonID| Name|Score|
+--------+-----+-----+
|       1|Alice|   88|
|       2|  Bob|   11|
|       3|Devis|   27|
|       4| Tara|   45|
|       5| John|   63|
+--------+-----+-----+



In [0]:
friend_join=friend_df.alias("a").join(person_df.alias("b"),col("a.fid")==col("b.personid"),"inner").select(col("a.pid"),("a.fid"),("b.score"))\
    .groupBy("pid").agg(count("*").alias("total_friend"),sum(col("score")).alias("total_score"))\
    .filter(col("total_score")>100)
person_info=friend_join.alias("a").join(person_df.alias("b"),col("a.pid")==col("b.personid"),"inner").select(col("a.*"),col("b.name")).show()

+---+------------+-----------+----+
|pid|total_friend|total_score|name|
+---+------------+-----------+----+
|  2|           2|        115| Bob|
|  4|           3|        101|Tara|
+---+------------+-----------+----+



In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Define schema for Trips
trips_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("client_id", IntegerType(), True),
    StructField("driver_id", IntegerType(), True),
    StructField("city_id", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("request_at", StringType(), True)
])

# Define data for Trips
trips_data = [
    (1, 1, 10, 1, 'completed', '2013-10-01'),
    (2, 2, 11, 1, 'cancelled_by_driver', '2013-10-01'),
    (3, 3, 12, 6, 'completed', '2013-10-01'),
    (4, 4, 13, 6, 'cancelled_by_client', '2013-10-01'),
    (5, 1, 10, 1, 'completed', '2013-10-02'),
    (6, 2, 11, 6, 'completed', '2013-10-02'),
    (7, 3, 12, 6, 'completed', '2013-10-02'),
    (8, 2, 12, 12, 'completed', '2013-10-03'),
    (9, 3, 10, 12, 'completed', '2013-10-03'),
    (10, 4, 13, 12, 'cancelled_by_driver', '2013-10-03')
]

# Create DataFrame
trips_df = spark.createDataFrame(trips_data, schema=trips_schema)
trips_df.show()


+---+---------+---------+-------+-------------------+----------+
| id|client_id|driver_id|city_id|             status|request_at|
+---+---------+---------+-------+-------------------+----------+
|  1|        1|       10|      1|          completed|2013-10-01|
|  2|        2|       11|      1|cancelled_by_driver|2013-10-01|
|  3|        3|       12|      6|          completed|2013-10-01|
|  4|        4|       13|      6|cancelled_by_client|2013-10-01|
|  5|        1|       10|      1|          completed|2013-10-02|
|  6|        2|       11|      6|          completed|2013-10-02|
|  7|        3|       12|      6|          completed|2013-10-02|
|  8|        2|       12|     12|          completed|2013-10-03|
|  9|        3|       10|     12|          completed|2013-10-03|
| 10|        4|       13|     12|cancelled_by_driver|2013-10-03|
+---+---------+---------+-------+-------------------+----------+



In [0]:
# Define schema for Users
users_schema = StructType([
    StructField("users_id", IntegerType(), True),
    StructField("banned", StringType(), True),
    StructField("role", StringType(), True)
])

# Define data for Users
users_data = [
    (1, 'No', 'client'),
    (2, 'Yes', 'client'),
    (3, 'No', 'client'),
    (4, 'No', 'client'),
    (10, 'No', 'driver'),
    (11, 'No', 'driver'),
    (12, 'No', 'driver'),
    (13, 'No', 'driver')
]

# Create DataFrame
users_df = spark.createDataFrame(users_data, schema=users_schema)
users_df.show()


+--------+------+------+
|users_id|banned|  role|
+--------+------+------+
|       1|    No|client|
|       2|   Yes|client|
|       3|    No|client|
|       4|    No|client|
|      10|    No|driver|
|      11|    No|driver|
|      12|    No|driver|
|      13|    No|driver|
+--------+------+------+



In [0]:
active_user=users_df.filter(col("banned")=='No')
trips_info=trips_df.alias("a").join(active_user.alias("b"),col("a.client_id")==col("b.users_id"),"inner") \
    .join(active_user.alias("c"),col("a.driver_id")==col("c.users_id"),"inner") \
    .withColumn("cancel_status",when((col("status")=='cancelled_by_driver') | (col("status")=='cancelled_by_client'),1).otherwise(0))\
    .groupBy("request_at").agg(sum(col("cancel_status")).alias("total_cancel"),count("*").alias("total_status"),round((sum(col("cancel_status"))*1.0  / count("*"))*100, 2).alias("cancel_percent")).show()

+----------+------------+------------+--------------+
|request_at|total_cancel|total_status|cancel_percent|
+----------+------------+------------+--------------+
|2013-10-01|           1|           3|         33.33|
|2013-10-02|           0|           2|           0.0|
|2013-10-03|           1|           2|          50.0|
+----------+------------+------------+--------------+



In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
data=[
    (101,["p1","p2","p5","p6"]),
    (102,["p3"]),
    (103,["p1","p4"]),
    (104,["p1","p2","p3"])
]
schema=StructType([
    StructField("emp_id",IntegerType(),True),
    StructField("project",ArrayType(StringType()),True)
])
df= spark.createDataFrame(data,schema)
df.show(truncate=False)
explode_df=df.withColumn("project",explode(col("project")))
explode_df.groupBy("project").agg(countDistinct(col("emp_id")).alias("emp_count")).show()

+------+----------------+
|emp_id|project         |
+------+----------------+
|101   |[p1, p2, p5, p6]|
|102   |[p3]            |
|103   |[p1, p4]        |
|104   |[p1, p2, p3]    |
+------+----------------+

+-------+---------+
|project|emp_count|
+-------+---------+
|     p6|        1|
|     p2|        2|
|     p1|        3|
|     p4|        1|
|     p5|        1|
|     p3|        2|
+-------+---------+



In [0]:
from pyspark.sql.functions import  *
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Create players DataFrame
players_data = [
    (15, 1),
    (25, 1),
    (30, 1),
    (45, 1),
    (10, 2),
    (35, 2),
    (50, 2),
    (20, 3),
    (40, 3)
]

players_columns = ["player_id", "group_id"]
players_df = spark.createDataFrame(players_data, players_columns)

# Show players DataFrame
players_df.show()

# Create matches DataFrame
matches_data = [
    (1, 15, 45, 3, 0),
    (2, 30, 25, 1, 2),
    (3, 30, 15, 2, 0),
    (4, 40, 20, 5, 2),
    (5, 35, 50, 1, 1)
]

matches_columns = ["match_id", "first_player", "second_player", "first_score", "second_score"]
matches_df = spark.createDataFrame(matches_data, matches_columns)

# Show matches DataFrame
matches_df.show()


+---------+--------+
|player_id|group_id|
+---------+--------+
|       15|       1|
|       25|       1|
|       30|       1|
|       45|       1|
|       10|       2|
|       35|       2|
|       50|       2|
|       20|       3|
|       40|       3|
+---------+--------+

+--------+------------+-------------+-----------+------------+
|match_id|first_player|second_player|first_score|second_score|
+--------+------------+-------------+-----------+------------+
|       1|          15|           45|          3|           0|
|       2|          30|           25|          1|           2|
|       3|          30|           15|          2|           0|
|       4|          40|           20|          5|           2|
|       5|          35|           50|          1|           1|
+--------+------------+-------------+-----------+------------+



In [0]:
first_player=matches_df.select(col("first_player").alias("player"),col("first_score").alias("score"))
second_player=matches_df.select(col("second_player").alias("player"),col("second_score").alias("score"))
union_df=first_player.union(second_player)
union_df.groupBy("player").agg(sum(col("score")).alias("total_score"))
join_df=union_df.alias("a").join(players_df.alias("b"),col("a.player")==col("b.player_id"),"inner").select(col("player"),col("score"),col("group_id"))
w=Window.partitionBy(col("group_id")).orderBy(col("score").desc(),col("player").asc())
join_df.withColumn("rnk",dense_rank().over(w)).filter(col("rnk")==1).drop(col("rnk")).show()

+------+-----+--------+
|player|score|group_id|
+------+-----+--------+
|    15|    3|       1|
|    35|    1|       2|
|    40|    5|       3|
+------+-----+--------+



In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Create Users DataFrame
users_data = [
    (1, '2019-01-01', 'Lenovo'),
    (2, '2019-02-09', 'Samsung'),
    (3, '2019-01-19', 'LG'),
    (4, '2019-05-21', 'HP')
]
users_columns = ["user_id", "join_date", "favorite_brand"]
users_df = spark.createDataFrame(users_data, users_columns)

# Create Items DataFrame
items_data = [
    (1, 'Samsung'),
    (2, 'Lenovo'),
    (3, 'LG'),
    (4, 'HP')
]
items_columns = ["item_id", "item_brand"]
items_df = spark.createDataFrame(items_data, items_columns)

# Create Orders DataFrame
orders_data = [
    (1, '2019-08-01', 4, 1, 2),
    (2, '2019-08-02', 2, 1, 3),
    (3, '2019-08-03', 3, 2, 3),
    (4, '2019-08-04', 1, 4, 2),
    (5, '2019-08-04', 1, 3, 4),
    (6, '2019-08-05', 2, 2, 4)
]
orders_columns = ["order_id", "order_date", "item_id", "buyer_id", "seller_id"]
orders_df = spark.createDataFrame(orders_data, orders_columns)



In [0]:
w=Window.partitionBy(col("seller_id")).orderBy(col("order_date").asc())
second_selling_items=orders_df.withColumn("rnk", dense_rank().over(w)) \
    .filter(col("rnk")==2) \
    .select(col("seller_id"),col("item_id"))

join_df=users_df.alias("a").join(second_selling_items.alias("b"),col("a.user_id")==col("b.seller_id"),"left") \
    .join(items_df.alias("c"),col("b.item_id")==col("c.item_id"),"left") \
    .select(col("user_id"),col("item_brand").alias("second_selling_brand"),col("a.favorite_brand"),when( col("a.favorite_brand")==col("c.item_brand"),"yes").otherwise("No").alias("same_brand"))
join_df.show()


+-------+--------------------+--------------+----------+
|user_id|second_selling_brand|favorite_brand|same_brand|
+-------+--------------------+--------------+----------+
|      1|                NULL|        Lenovo|        No|
|      2|             Samsung|       Samsung|       yes|
|      3|                  LG|            LG|       yes|
|      4|              Lenovo|            HP|        No|
+-------+--------------------+--------------+----------+



In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import datetime


# Define the schema using StructType
tasks_schema = StructType([
    StructField("date_value", DateType(), True),
    StructField("state", StringType(), True)
])

# Define the data for tasks
tasks_data = [
    (datetime.date(2019, 1, 1), 'success'),
    (datetime.date(2019, 1, 2), 'success'),
    (datetime.date(2019, 1, 3), 'success'),
    (datetime.date(2019, 1, 4), 'fail'),
    (datetime.date(2019, 1, 5), 'fail'),
    (datetime.date(2019, 1, 6), 'success')
]

# Create the DataFrame with the specified schema
tasks_df = spark.createDataFrame(tasks_data, schema=tasks_schema)

# Show the tasks DataFrame
tasks_df.show()


+----------+-------+
|date_value|  state|
+----------+-------+
|2019-01-01|success|
|2019-01-02|success|
|2019-01-03|success|
|2019-01-04|   fail|
|2019-01-05|   fail|
|2019-01-06|success|
+----------+-------+



In [0]:
w=Window.orderBy(col("date_value").asc())
w1=Window.partitionBy(col("state")).orderBy(col("date_value").asc())
diff=tasks_df.withColumn("rnk",row_number().over(w)-row_number().over(w1))
diff.groupBy(col("state"),col("rnk")).agg(min(col("date_value")).alias("start_date"),max(col("date_value")).alias("end_date")).drop("rnk").show()

diff1=tasks_df.withColumn("rnk",date_add("date_value",-1*row_number().over(w1)))
diff1.groupBy(col("state"),col("rnk")).agg(min(col("date_value")).alias("start_date"),max(col("date_value")).alias("end_date")).orderBy(col("state")).drop("rnk").show()


+-------+----------+----------+
|  state|start_date|  end_date|
+-------+----------+----------+
|success|2019-01-01|2019-01-03|
|   fail|2019-01-04|2019-01-05|
|success|2019-01-06|2019-01-06|
+-------+----------+----------+

+-------+----------+----------+
|  state|start_date|  end_date|
+-------+----------+----------+
|   fail|2019-01-04|2019-01-05|
|success|2019-01-01|2019-01-03|
|success|2019-01-06|2019-01-06|
+-------+----------+----------+



In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import datetime



# Define the schema using StructType
spending_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("spend_date", StringType(), True),
    StructField("platform", StringType(), True),
    StructField("amount", IntegerType(), True)
])

# Define the data for spending
spending_data = [
    (1, '2019-07-01', 'mobile', 100),
    (1, '2019-07-01', 'desktop', 100),
    (2, '2019-07-01', 'mobile', 100),
    (2, '2019-07-02', 'mobile', 100),
    (3, '2019-07-01', 'desktop', 100),
    (3, '2019-07-02', 'desktop', 100)
]

# Create the DataFrame with the specified schema
spending_df = spark.createDataFrame(spending_data, schema=spending_schema)

spending_df.withColumn("spend_date", to_date(spending_df["spend_date"], "yyyy-MM-dd")).show()





+-------+----------+--------+------+
|user_id|spend_date|platform|amount|
+-------+----------+--------+------+
|      1|2019-07-01|  mobile|   100|
|      1|2019-07-01| desktop|   100|
|      2|2019-07-01|  mobile|   100|
|      2|2019-07-02|  mobile|   100|
|      3|2019-07-01| desktop|   100|
|      3|2019-07-02| desktop|   100|
+-------+----------+--------+------+



In [0]:
spending_df1 = spending_df.groupBy("user_id", "spend_date").agg(
    sum(col("amount")).alias("amount"),
    max(col("platform")).alias("platform"),
    countDistinct("platform").alias("cnt")
).filter(col("cnt") == 1)
spending_df2 = spending_df.groupBy("user_id", "spend_date").agg(
    sum(col("amount")).alias("amount"),
    lit("both").alias("platform"),
    countDistinct("platform").alias("cnt")
).filter(col("cnt") == 2)
spending_df3=spending_df.select(lit(None).alias("user_id"),col("spend_date"),lit(0).alias("amount"),lit("both").alias("platform"),lit(0).alias("cnt"))

union_df=spending_df1.union(spending_df2).union(spending_df3)
union_df.groupBy("spend_date","platform").agg(sum(col("amount")).alias("amount"),count(col("user_id")).alias("count_user")).show()

+----------+--------+------+----------+
|spend_date|platform|amount|count_user|
+----------+--------+------+----------+
|2019-07-02|  mobile|   100|         1|
|2019-07-01| desktop|   100|         1|
|2019-07-01|  mobile|   100|         1|
|2019-07-02| desktop|   100|         1|
|2019-07-01|    both|   200|         1|
|2019-07-02|    both|     0|         0|
+----------+--------+------+----------+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Define schema for users table
users_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("join_date", StringType(), True)
])

# Define schema for events table
events_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("type", StringType(), True),
    StructField("access_date", StringType(), True)
])

# Data for users table
users_data = [
    (1, 'Jon', '2025-02-04'),
    (2, 'Jane', '2025-02-14'),
    (3, 'Jill', '2025-02-15'),
    (4, 'Josh', '2025-02-15'),
    (5, 'Jean', '2025-02-16'),
    (6, 'Justin', '2025-02-17'),
    (7, 'Jeremy', '2025-02-18')
]

# Data for events table
events_data = [
    (1, 'Pay', '2025-03-01'),
    (2, 'Music', '2025-03-02'),
    (2, 'P', '2025-03-12'),
    (3, 'Music', '2025-03-15'),
    (4, 'Music', '2025-03-15'),
    (1, 'P', '2025-03-16'),
    (3, 'P', '2025-03-22')
]

# Create DataFrames from the data and schemas
users_df = spark.createDataFrame(users_data, schema=users_schema)
events_df = spark.createDataFrame(events_data, schema=events_schema)

# Convert 'join_date' and 'access_date' to DateType
users_df.withColumn("join_date", to_date(users_df["join_date"], "yyyy-MM-dd")).show()
events_df.withColumn("access_date", to_date(events_df["access_date"], "yyyy-MM-dd")).show()


+-------+------+----------+
|user_id|  name| join_date|
+-------+------+----------+
|      1|   Jon|2025-02-04|
|      2|  Jane|2025-02-14|
|      3|  Jill|2025-02-15|
|      4|  Josh|2025-02-15|
|      5|  Jean|2025-02-16|
|      6|Justin|2025-02-17|
|      7|Jeremy|2025-02-18|
+-------+------+----------+

+-------+-----+-----------+
|user_id| type|access_date|
+-------+-----+-----------+
|      1|  Pay| 2025-03-01|
|      2|Music| 2025-03-02|
|      2|    P| 2025-03-12|
|      3|Music| 2025-03-15|
|      4|Music| 2025-03-15|
|      1|    P| 2025-03-16|
|      3|    P| 2025-03-22|
+-------+-----+-----------+



In [0]:
music_events=events_df.filter(col("type")=="Music").select("user_id")
music_event_ids = [row.user_id for row in music_events.collect()]
music_users=users_df.filter(col("user_id").isin(music_event_ids))
primeusers=music_users.alias("a").join(events_df.alias("b"),(col("a.user_id")==col("b.user_id")) & (col("b.type")=='P'),"left").withColumn( "prime_user",when(datediff(col("b.access_date"),col("a.join_date"))<=30,col("b.user_id")).otherwise(None)).agg(count("*").alias("total_music_user"),count(col("prime_user")).alias("prime_user")) \
    .withColumn("fraction",(1.0*col("prime_user")/col("total_music_user"))*100).show()

+----------------+----------+-----------------+
|total_music_user|prime_user|         fraction|
+----------------+----------+-----------------+
|               3|         1|33.33333333333333|
+----------------+----------+-----------------+



In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import date



# Define schema
schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("cust_id", IntegerType(), True),
    StructField("order_date", DateType(), True),
    StructField("amount", IntegerType(), True)
])

# Create data
data = [
    (1, 1, date(2020, 1, 15), 150),
    (2, 1, date(2020, 2, 10), 150),
    (3, 2, date(2020, 1, 16), 150),
    (4, 2, date(2020, 2, 25), 150),
    (5, 3, date(2020, 1, 10), 150),
    (6, 3, date(2020, 2, 20), 150),
    (7, 4, date(2020, 1, 20), 150),
    (8, 5, date(2020, 2, 20), 150)
]

# Create DataFrame
transactions = spark.createDataFrame(data, schema)

# Show DataFrame
transactions.show()


+--------+-------+----------+------+
|order_id|cust_id|order_date|amount|
+--------+-------+----------+------+
|       1|      1|2020-01-15|   150|
|       2|      1|2020-02-10|   150|
|       3|      2|2020-01-16|   150|
|       4|      2|2020-02-25|   150|
|       5|      3|2020-01-10|   150|
|       6|      3|2020-02-20|   150|
|       7|      4|2020-01-20|   150|
|       8|      5|2020-02-20|   150|
+--------+-------+----------+------+



In [0]:
w=Window.partitionBy(col("cust_id")).orderBy(col("order_date"))
df1=transactions.withColumn("month",month(col("order_date"))).withColumn("prevous_date",lag(col("order_date"),1,col("order_date")).over(w))
df1.withColumn("diff" , when(month((col("order_date")))- month(col("prevous_date"))==1,1).otherwise(0))\
.groupBy(col("month")).agg(sum(col("diff")).alias("tital_retention")).show()


w=Window.partitionBy(col("cust_id")).orderBy(col("order_date"))
df1=transactions.withColumn("month",month(col("order_date"))).withColumn("next_date",lead(col("order_date")).over(w))
df1.withColumn("diff",when(
    month(col("next_date"))-month(col("order_date"))==1,0
        ).otherwise(1)
            ) \
    .groupBy(col("month"))\
    .agg(sum(col("diff")).alias("total_churn")).show()

+-----+---------------+
|month|tital_retention|
+-----+---------------+
|    1|              0|
|    2|              3|
+-----+---------------+

+-----+-----------+
|month|total_churn|
+-----+-----------+
|    1|          1|
|    2|          4|
+-----+-----------+



In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *
from datetime import date

# Define schema
schema = StructType([
    StructField("username", StringType(), True),
    StructField("activity", StringType(), True),
    StructField("startDate", DateType(), True),
    StructField("endDate", DateType(), True)
])

# Create data
data = [
    ("Alice", "Travel", date(2020, 2, 12), date(2020, 2, 20)),
    ("Alice", "Dancing", date(2020, 2, 21), date(2020, 2, 23)),
    ("Alice", "Travel", date(2020, 2, 24), date(2020, 2, 28)),
    ("Bob", "Travel", date(2020, 2, 11), date(2020, 2, 18))
]

# Create DataFrame
df = spark.createDataFrame(data, schema)

# Show DataFrame
df.show()


+--------+--------+----------+----------+
|username|activity| startDate|   endDate|
+--------+--------+----------+----------+
|   Alice|  Travel|2020-02-12|2020-02-20|
|   Alice| Dancing|2020-02-21|2020-02-23|
|   Alice|  Travel|2020-02-24|2020-02-28|
|     Bob|  Travel|2020-02-11|2020-02-18|
+--------+--------+----------+----------+



In [0]:
w1=Window.partitionBy(col("username")).orderBy(col("startDate").desc())
w2=Window.partitionBy(col("username"))
df.withColumn("rank",dense_rank().over(w1)).withColumn("count",count("*").over(w2))\
    .filter((col("rank")==2) | (col("count")==1)).drop("rank","count").show()

+--------+--------+----------+----------+
|username|activity| startDate|   endDate|
+--------+--------+----------+----------+
|   Alice| Dancing|2020-02-21|2020-02-23|
|     Bob|  Travel|2020-02-11|2020-02-18|
+--------+--------+----------+----------+



In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *
from datetime import datetime

# Define schema for billings
billing_schema = StructType([
    StructField("emp_name", StringType(), True),
    StructField("bill_date", DateType(), True),
    StructField("bill_rate", IntegerType(), True)
])

# Define data for billings
billing_data = [
    ("Sachin", datetime.strptime("01-JAN-1990", "%d-%b-%Y").date(), 25),
    ("Sehwag", datetime.strptime("01-JAN-1992", "%d-%b-%Y").date(), 15),
    ("Sehwag", datetime.strptime("01-JAN-1990", "%d-%b-%Y").date(), 16),
    ("Sehwag", datetime.strptime("01-JAN-1991", "%d-%b-%Y").date(), 17),
    ("Dhoni",  datetime.strptime("01-JAN-1989", "%d-%b-%Y").date(), 20),
    ("Sachin", datetime.strptime("05-FEB-1991", "%d-%b-%Y").date(), 30)
]

# Create billings DataFrame
billings_df = spark.createDataFrame(billing_data, schema=billing_schema)
# Define schema for HoursWorked
hours_schema = StructType([
    StructField("emp_name", StringType(), True),
    StructField("work_date", DateType(), True),
    StructField("bill_hrs", IntegerType(), True)
])

# Define data for HoursWorked
hours_data = [
    ("Sachin", datetime.strptime("01-JUL-1990", "%d-%b-%Y").date(), 3),
    ("Sachin", datetime.strptime("01-AUG-1990", "%d-%b-%Y").date(), 5),
    ("Sehwag", datetime.strptime("01-JUL-1990", "%d-%b-%Y").date(), 2),
    ("Sehwag", datetime.strptime("01-JUL-1991", "%d-%b-%Y").date(), 5),
    ("Sehwag", datetime.strptime("01-SEP-1992", "%d-%b-%Y").date(), 4),
    ("Sachin", datetime.strptime("01-JUL-1991", "%d-%b-%Y").date(), 4)
]

# Create HoursWorked DataFrame
hours_df = spark.createDataFrame(hours_data, schema=hours_schema)

# Show
hours_df.show()
billings_df.show()

+--------+----------+--------+
|emp_name| work_date|bill_hrs|
+--------+----------+--------+
|  Sachin|1990-07-01|       3|
|  Sachin|1990-08-01|       5|
|  Sehwag|1990-07-01|       2|
|  Sehwag|1991-07-01|       5|
|  Sehwag|1992-09-01|       4|
|  Sachin|1991-07-01|       4|
+--------+----------+--------+

+--------+----------+---------+
|emp_name| bill_date|bill_rate|
+--------+----------+---------+
|  Sachin|1990-01-01|       25|
|  Sehwag|1992-01-01|       15|
|  Sehwag|1990-01-01|       16|
|  Sehwag|1991-01-01|       17|
|   Dhoni|1989-01-01|       20|
|  Sachin|1991-02-05|       30|
+--------+----------+---------+



In [0]:

w=Window.partitionBy(col("emp_name")).orderBy(col("bill_date"))
df1=billings_df.withColumn("next_date",lead(col("bill_date"),1,'9999-12-31').over(w))
df1.alias("a").join(hours_df.alias("b"),
    (
        (col("a.emp_name")==col("b.emp_name")) & 
        (col("b.work_date").between(col("a.bill_date"),col("a.next_date")))
    ),"inner") \
    .groupBy(col("a.emp_name")) \
    .agg(sum(col("a.bill_rate")*col("b.bill_hrs")).alias("total")).show()

+--------+-----+
|emp_name|total|
+--------+-----+
|  Sachin|  320|
|  Sehwag|  177|
+--------+-----+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.types import *
from pyspark.sql.functions import *
from datetime import datetime

# Define schema for the bms table
schema = StructType([
    StructField("seat_no", IntegerType(), True),
    StructField("is_empty", StringType(), True)
])

# Create data (the data you want to insert into the table)
data = [
    (1, 'N'),
    (2, 'Y'),
    (3, 'N'),
    (4, 'Y'),
    (5, 'Y'),
    (6, 'Y'),
    (7, 'N'),
    (8, 'Y'),
    (9, 'Y'),
    (10, 'Y'),
    (11, 'Y'),
    (12, 'N'),
    (13, 'Y'),
    (14, 'Y')
]

# Create a DataFrame based on the schema and data
bms_df = spark.createDataFrame(data, schema)

# Show the table content
bms_df.show()


+-------+--------+
|seat_no|is_empty|
+-------+--------+
|      1|       N|
|      2|       Y|
|      3|       N|
|      4|       Y|
|      5|       Y|
|      6|       Y|
|      7|       N|
|      8|       Y|
|      9|       Y|
|     10|       Y|
|     11|       Y|
|     12|       N|
|     13|       Y|
|     14|       Y|
+-------+--------+



In [0]:
w=Window.partitionBy(col("is_empty")).orderBy(col("seat_no"))
bms_df.filter(col("is_empty")=='Y').withColumn("grouping",col("seat_no")-row_number().over(w))\
    .withColumn("count",count("*").over(Window.partitionBy(col("grouping")))) \
    .filter(col("count")>=3) \
    .select(col("seat_no")).show()


+-------+
|seat_no|
+-------+
|      4|
|      5|
|      6|
|      8|
|      9|
|     10|
|     11|
+-------+



In [0]:
window_spec = Window.orderBy("seat_no")

# Calculate prev_row2, next_row2, and current_row using window functions
method2_df = bms_df.withColumn(
    "prev_row2", 
    sum(when(col("is_empty") == 'Y', 1).otherwise(0)).over(window_spec.rowsBetween(-2, 0))
).withColumn(
    "next_row2", 
    sum(when(col("is_empty") == 'Y', 1).otherwise(0)).over(window_spec.rowsBetween(0, 2))
).withColumn(
    "current_row", 
    sum(when(col("is_empty") == 'Y', 1).otherwise(0)).over(window_spec.rowsBetween(-1, 1))
)

# Filter the rows where 3 is in any of prev_row2, current_row, or next_row2
filtered_df = method2_df.filter(
    (col("prev_row2") == 3) | 
    (col("current_row") == 3) | 
    (col("next_row2") == 3)
)

# Select seat_no as per the final query
result_df = filtered_df.select("seat_no")

# Show the result
result_df.show()

+-------+
|seat_no|
+-------+
|      4|
|      5|
|      6|
|      8|
|      9|
|     10|
|     11|
+-------+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.types import *
from pyspark.sql.functions import *
from datetime import datetime

# Define schema for the 'covid' table
schema = StructType([
    StructField("city", StringType(), True),
    StructField("days", DateType(), True),
    StructField("cases", IntegerType(), True)
])

# Create data (equivalent to your SQL inserts)
data = [
    ('DELHI', datetime.strptime('2022-01-01', '%Y-%m-%d'), 100),
    ('DELHI', datetime.strptime('2022-01-02', '%Y-%m-%d'), 200),
    ('DELHI', datetime.strptime('2022-01-03', '%Y-%m-%d'), 300),
    ('MUMBAI', datetime.strptime('2022-01-01', '%Y-%m-%d'), 100),
    ('MUMBAI', datetime.strptime('2022-01-02', '%Y-%m-%d'), 100),
    ('MUMBAI', datetime.strptime('2022-01-03', '%Y-%m-%d'), 300),
    ('CHENNAI', datetime.strptime('2022-01-01', '%Y-%m-%d'), 100),
    ('CHENNAI', datetime.strptime('2022-01-02', '%Y-%m-%d'), 200),
    ('CHENNAI', datetime.strptime('2022-01-03', '%Y-%m-%d'), 150),
    ('BANGALORE', datetime.strptime('2022-01-01', '%Y-%m-%d'), 100),
    ('BANGALORE', datetime.strptime('2022-01-02', '%Y-%m-%d'), 300),
    ('BANGALORE', datetime.strptime('2022-01-03', '%Y-%m-%d'), 200),
    ('BANGALORE', datetime.strptime('2022-01-04', '%Y-%m-%d'), 400)
]

# Create DataFrame based on the schema and data
covid_df = spark.createDataFrame(data, schema)

# Show the data (equivalent to SELECT * FROM covid)
display(covid_df)

city,days,cases
DELHI,2022-01-01,100
DELHI,2022-01-02,200
DELHI,2022-01-03,300
MUMBAI,2022-01-01,100
MUMBAI,2022-01-02,100
MUMBAI,2022-01-03,300
CHENNAI,2022-01-01,100
CHENNAI,2022-01-02,200
CHENNAI,2022-01-03,150
BANGALORE,2022-01-01,100


In [0]:
w=Window.partitionBy(col("city")).orderBy(col("days"))
covid_df.withColumn("previous_case",lag(col("cases"),1,0).over(w))\
.withColumn("check_case",when(col("cases")>col("previous_case"),0).otherwise(lit(1))) \
.groupBy(col("city")).agg(sum(col("check_case")).alias("days")).filter(col("days")==0).drop(col("day")).show()

+-----+----+
| city|days|
+-----+----+
|DELHI|   0|
+-----+----+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.types import *
from pyspark.sql.functions import *
from datetime import datetime
# Define schema for company_users
schema = StructType([
    StructField("company_id", IntegerType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("language", StringType(), True)
])

# Data to insert
data = [
    (1, 1, 'English'),
    (1, 1, 'German'),
    (1, 2, 'English'),
    (1, 3, 'German'),
    (1, 3, 'English'),
    (1, 4, 'English'),
    (2, 5, 'English'),
    (2, 5, 'German'),
    (2, 5, 'Spanish'),
    (2, 6, 'German'),
    (2, 6, 'Spanish'),
    (2, 7, 'English')
]

# Create DataFrame
company_users_df = spark.createDataFrame(data, schema)

# Show the DataFrame
company_users_df.show()

+----------+-------+--------+
|company_id|user_id|language|
+----------+-------+--------+
|         1|      1| English|
|         1|      1|  German|
|         1|      2| English|
|         1|      3|  German|
|         1|      3| English|
|         1|      4| English|
|         2|      5| English|
|         2|      5|  German|
|         2|      5| Spanish|
|         2|      6|  German|
|         2|      6| Spanish|
|         2|      7| English|
+----------+-------+--------+



In [0]:
company_users_df.filter(col("language").isin("English","German")).groupBy(col("company_id"),col("user_id")).agg(count(col("user_id")).alias("total_users")).filter("total_users=2").groupBy("company_id").agg(count(col("company_id")).alias("total_user")).filter(col("total_user")==2).show()

company_users_df.withColumn("flag",when(col("language")=="English",lit(1)).when(col("language")=="German",lit(2)).otherwise(lit(0))).groupBy(col("company_id"),col("user_id")).agg(sum(col("flag")).alias("sum_flag")).filter(col("sum_flag")==3)\
.groupBy("company_id").agg(count(col("company_id")).alias("total_user")).filter(col("total_user")==2).show()

+----------+----------+
|company_id|total_user|
+----------+----------+
|         1|         2|
+----------+----------+

+----------+----------+
|company_id|total_user|
+----------+----------+
|         1|         2|
+----------+----------+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.types import *
from pyspark.sql.functions import *
from datetime import datetime


# Define schema for products
products_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("cost", IntegerType(), True)
])

# Insert data
products_data = [
    ('P1', 200),
    ('P2', 300),
    ('P3', 500),
    ('P4', 800)
]

# Create DataFrame
products_df = spark.createDataFrame(products_data, schema=products_schema)

# Show the data
print("Products Table:")
products_df.show()

# ----------------------------
# Customer Budget Table
# ----------------------------

# Define schema for customer_budget
budget_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("budget", IntegerType(), True)
])

# Insert data
budget_data = [
    (100, 400),
    (200, 800),
    (300, 1500)
]

# Create DataFrame
budget_df = spark.createDataFrame(budget_data, schema=budget_schema)

# Show the data
print("Customer Budget Table:")
budget_df.show()


Products Table:
+----------+----+
|product_id|cost|
+----------+----+
|        P1| 200|
|        P2| 300|
|        P3| 500|
|        P4| 800|
+----------+----+

Customer Budget Table:
+-----------+------+
|customer_id|budget|
+-----------+------+
|        100|   400|
|        200|   800|
|        300|  1500|
+-----------+------+



In [0]:
pd=products_df.withColumn("running_sum",sum(col("cost")).over(Window.orderBy(col("cost"))))
# budget_df.alias("a").join(pd.alias("b"),col("b.running_sum")<col("a.budget")).show()
budget_df.alias("a").join(
    pd,
    pd["running_sum"] < col("a.budget")
).groupBy("customer_id").agg(count("product_id").alias("count"),concat_ws(',',collect_list("product_id")).alias("product_list")).show()

+-----------+-----+------------+
|customer_id|count|product_list|
+-----------+-----+------------+
|        100|    1|          P1|
|        200|    2|       P2,P1|
|        300|    3|    P3,P2,P1|
+-----------+-----+------------+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Sample data
data = [
    ('2020-04-01', 'Avinash', 'Vibhor', 10),
    ('2020-04-01', 'Vibhor', 'Avinash', 20),
    ('2020-04-01', 'Avinash', 'Pawan', 30),
    ('2020-04-01', 'Pawan', 'Avinash', 20),
    ('2020-04-01', 'Vibhor', 'Pawan', 5),
    ('2020-04-01', 'Pawan', 'Vibhor', 8),
    ('2020-04-01', 'Vibhor', 'Deepak', 50)
]

columns = ["sms_date", "sender", "receiver", "sms_no"]

# Create DataFrame
subscriber_df = spark.createDataFrame(data, columns)

# Add p1 and p2 (ordered sender-receiver pair)
cte_df = subscriber_df.withColumn(
    "p1", when(col("sender") < col("receiver"), col("sender")).otherwise(col("receiver"))
).withColumn(
    "p2", when(col("sender") > col("receiver"), col("sender")).otherwise(col("receiver"))
)

# Aggregate SMS counts between user pairs
result_df = cte_df.groupBy("sms_date", "p1", "p2").agg(
    sum("sms_no").alias("total_sms")
)

# Show result
result_df.show(truncate=False)

df1 = subscriber_df.filter(col("sender") < col("receiver"))
df2 = subscriber_df.filter(col("sender") > col("receiver")) \
    .select(
        col("sms_date"),
        col("receiver").alias("sender"),
        col("sender").alias("receiver"),
        col("sms_no")
    )

# Step 5: UNION ALL
cte_df = df1.union(df2)

# Step 6: GROUP BY sender, receiver, sms_date and SUM(sms_no)
result_df = cte_df.groupBy("sender", "receiver", "sms_date") \
    .agg(sum(col("sms_no")).alias("total_sms"))
result_df.show(truncate=False)


+----------+-------+------+---------+
|sms_date  |p1     |p2    |total_sms|
+----------+-------+------+---------+
|2020-04-01|Avinash|Vibhor|30       |
|2020-04-01|Avinash|Pawan |50       |
|2020-04-01|Pawan  |Vibhor|13       |
|2020-04-01|Deepak |Vibhor|50       |
+----------+-------+------+---------+

+-------+--------+----------+---------+
|sender |receiver|sms_date  |total_sms|
+-------+--------+----------+---------+
|Avinash|Vibhor  |2020-04-01|30       |
|Avinash|Pawan   |2020-04-01|50       |
|Pawan  |Vibhor  |2020-04-01|13       |
|Deepak |Vibhor  |2020-04-01|50       |
+-------+--------+----------+---------+

